# Data Structure Validation

Validates the structure, dimensions, labels, and mergeability of all 8 LeRobot datasets.

In [2]:
!pip install datasets pyarrow pandas numpy -q
print('Dependencies installed.')

Dependencies installed.


In [3]:
from datasets import load_dataset
import pandas as pd
import numpy as np

PASS = "\u2705 PASS"
FAIL = "\u274c FAIL"
results = []

def check(name, condition, detail=""):
    status = PASS if condition else FAIL
    results.append((status, name, detail))
    print(f"  {status}  {name}" + (f"  [{detail}]" if detail else ""))

---
## TEST 1: Load All Datasets

In [4]:
# Simulation datasets
SIM_DATASETS = [
    "lerobot/pusht",
    "lerobot/xarm_lift_medium",
    "lerobot/xarm_lift_medium_replay",
    "lerobot/xarm_push_medium",
    "lerobot/xarm_push_medium_replay",
]

# Real robot datasets (Berkeley / Columbia / NYU)
REAL_DATASETS = [
    "lerobot/berkeley_autolab_ur5",
    "lerobot/columbia_cairlab_pusht_real",
    "lerobot/nyu_door_opening_surprising_effectiveness",
]

ALL_DATASETS = SIM_DATASETS + REAL_DATASETS

loaded = {}
print("=== TEST 1: Dataset Loading ===")
for name in ALL_DATASETS:
    try:
        ds = load_dataset(name, split="train")
        loaded[name] = ds
        check(f"Load {name.split('/')[1][:40]}", True, f"{len(ds):,} rows")
    except Exception as e:
        check(f"Load {name.split('/')[1][:40]}", False, str(e)[:60])

total_rows = sum(len(ds) for ds in loaded.values())
print(f"\n  Total rows: {total_rows:,}")
check("Total >= 100,000 rows (course requirement)", total_rows >= 100_000, f"actual: {total_rows:,}")
check("Total >= 250,000 rows (8 datasets)", total_rows >= 250_000, f"actual: {total_rows:,}")

=== TEST 1: Dataset Loading ===


  ✅ PASS  Load pusht  [25,650 rows]
  ✅ PASS  Load xarm_lift_medium  [20,000 rows]
  ✅ PASS  Load xarm_lift_medium_replay  [20,000 rows]
  ✅ PASS  Load xarm_push_medium  [20,000 rows]
  ✅ PASS  Load xarm_push_medium_replay  [20,000 rows]
  ✅ PASS  Load berkeley_autolab_ur5  [97,939 rows]
  ✅ PASS  Load columbia_cairlab_pusht_real  [27,808 rows]
  ✅ PASS  Load nyu_door_opening_surprising_effectivenes  [20,405 rows]

  Total rows: 251,802
  ✅ PASS  Total >= 100,000 rows (course requirement)  [actual: 251,802]
  ✅ PASS  Total >= 250,000 rows (8 datasets)  [actual: 251,802]


---
## TEST 2: Column Structure Consistency

In [5]:
print("=== TEST 2: Column Structure ===")

REQUIRED_COLS = ["observation.state", "action", "episode_index",
                 "frame_index", "next.reward", "next.done"]

for name, ds in loaded.items():
    short = name.split("/")[1][:30]
    for col in REQUIRED_COLS:
        check(f"{short} has column {col}", col in ds.column_names)

=== TEST 2: Column Structure ===
  ✅ PASS  pusht has column observation.state
  ✅ PASS  pusht has column action
  ✅ PASS  pusht has column episode_index
  ✅ PASS  pusht has column frame_index
  ✅ PASS  pusht has column next.reward
  ✅ PASS  pusht has column next.done
  ✅ PASS  xarm_lift_medium has column observation.state
  ✅ PASS  xarm_lift_medium has column action
  ✅ PASS  xarm_lift_medium has column episode_index
  ✅ PASS  xarm_lift_medium has column frame_index
  ✅ PASS  xarm_lift_medium has column next.reward
  ✅ PASS  xarm_lift_medium has column next.done
  ✅ PASS  xarm_lift_medium_replay has column observation.state
  ✅ PASS  xarm_lift_medium_replay has column action
  ✅ PASS  xarm_lift_medium_replay has column episode_index
  ✅ PASS  xarm_lift_medium_replay has column frame_index
  ✅ PASS  xarm_lift_medium_replay has column next.reward
  ✅ PASS  xarm_lift_medium_replay has column next.done
  ✅ PASS  xarm_push_medium has column observation.state
  ✅ PASS  xarm_push_medium has c

---
## TEST 3: State/Action Dimensions per Dataset

In [6]:
print("=== TEST 3: Dimension Check ===")

dims = {}
for name, ds in loaded.items():
    row = ds[0]
    state_dim  = len(row["observation.state"])
    action_dim = len(row["action"])
    dims[name] = (state_dim, action_dim)
    short = name.split("/")[1]
    print(f"  {short:50s}  state={state_dim}D  action={action_dim}D")

# Simulation dimension checks
pusht_state = dims.get("lerobot/pusht", (0,0))[0]
xarm_state  = dims.get("lerobot/xarm_lift_medium", (0,0))[0]
check("pusht state=2D", pusht_state == 2, f"actual: {pusht_state}D")
check("xarm state=4D",  xarm_state  == 4, f"actual: {xarm_state}D")
check("All xarm state dimensions consistent",
      all(dims.get(f"lerobot/{n}", (0,0))[0] == 4 for n in
          ["xarm_lift_medium","xarm_lift_medium_replay",
           "xarm_push_medium","xarm_push_medium_replay"]))

# Real robot dimension checks (should all be: state=8D, action=7D)
real_names = [n for n in REAL_DATASETS if n in dims]
if real_names:
    real_states  = [dims[n][0] for n in real_names]
    real_actions = [dims[n][1] for n in real_names]
    check("Real robot state dimensions consistent", len(set(real_states)) == 1,
          f"dimensions: {real_states}")
    check("Real robot action dimensions consistent", len(set(real_actions)) == 1,
          f"dimensions: {real_actions}")
    check("Real robot state=8D", all(d == 8 for d in real_states),
          f"actual: {real_states}")
    check("Real robot action=7D", all(d == 7 for d in real_actions),
          f"actual: {real_actions}")

=== TEST 3: Dimension Check ===
  pusht                                               state=2D  action=2D
  xarm_lift_medium                                    state=4D  action=4D
  xarm_lift_medium_replay                             state=4D  action=4D
  xarm_push_medium                                    state=4D  action=3D
  xarm_push_medium_replay                             state=4D  action=3D
  berkeley_autolab_ur5                                state=8D  action=7D
  columbia_cairlab_pusht_real                         state=8D  action=7D
  nyu_door_opening_surprising_effectiveness           state=8D  action=7D
  ✅ PASS  pusht state=2D  [actual: 2D]
  ✅ PASS  xarm state=4D  [actual: 4D]
  ✅ PASS  All xarm state dimensions consistent
  ✅ PASS  Real robot state dimensions consistent  [dimensions: [8, 8, 8]]
  ✅ PASS  Real robot action dimensions consistent  [dimensions: [7, 7, 7]]
  ✅ PASS  Real robot state=8D  [actual: [8, 8, 8]]
  ✅ PASS  Real robot action=7D  [actual: [7, 7, 7]]


---
## TEST 4: Reward Label Ranges

In [7]:
print("=== TEST 4: Reward Labels ===")

for name, ds in loaded.items():
    rewards = [row["next.reward"] for row in ds.select(range(min(200, len(ds))))]
    r_min, r_max = min(rewards), max(rewards)
    short = name.split("/")[1][:30]
    check(f"{short} reward is numeric", all(isinstance(r, (int,float)) for r in rewards),
          f"range: [{r_min:.3f}, {r_max:.3f}]")

# xarm_push reward should be negative (distance metric)
for name in ["lerobot/xarm_push_medium", "lerobot/xarm_push_medium_replay"]:
    if name in loaded:
        ds = loaded[name]
        rewards = [row["next.reward"] for row in ds.select(range(min(200, len(ds))))]
        short = name.split("/")[1]
        check(f"{short} reward is negative (distance metric)",
              max(rewards) <= 0,
              f"max reward: {max(rewards):.4f}")

=== TEST 4: Reward Labels ===
  ✅ PASS  pusht reward is numeric  [range: [0.001, 0.873]]
  ✅ PASS  xarm_lift_medium reward is numeric  [range: [-0.035, 1.199]]
  ✅ PASS  xarm_lift_medium_replay reward is numeric  [range: [-0.034, 0.573]]
  ✅ PASS  xarm_push_medium reward is numeric  [range: [-0.979, -0.044]]
  ✅ PASS  xarm_push_medium_replay reward is numeric  [range: [-1.278, -0.382]]
  ✅ PASS  berkeley_autolab_ur5 reward is numeric  [range: [0.000, 1.000]]
  ✅ PASS  columbia_cairlab_pusht_real reward is numeric  [range: [0.000, 0.000]]
  ✅ PASS  nyu_door_opening_surprising_ef reward is numeric  [range: [0.000, 1.000]]
  ✅ PASS  xarm_push_medium reward is negative (distance metric)  [max reward: -0.0440]
  ✅ PASS  xarm_push_medium_replay reward is negative (distance metric)  [max reward: -0.3818]


---
## TEST 5: Aggregated Feature Extraction

Extract dimension-agnostic episode-level features using L2 norm aggregation.

In [8]:
def extract_episode_features(ds, source_name, n_episodes=20):
    """
    Aggregate frame-level data into episode-level, dimension-agnostic statistical features.

    Why use aggregated features instead of raw frames?
    - pusht state=2D, xarm state=4D, real=8D -- different dims cannot be concatenated directly
    - Aggregated statistics (mean, std, L2 norm) are valid for any dimensionality
    - Episode-level features capture meaningful trajectory patterns for prediction
    """
    episodes_data = []
    ep_indices = sorted(set(row["episode_index"]
                            for row in ds.select(range(min(500, len(ds))))))[:n_episodes]

    for ep_idx in ep_indices:
        frames = [row for row in ds if row["episode_index"] == ep_idx]
        if not frames:
            continue

        states  = np.array([row["observation.state"] for row in frames])
        actions = np.array([row["action"] for row in frames])
        rewards = [row["next.reward"] for row in frames]

        episodes_data.append({
            "source":            source_name,
            "episode_index":     ep_idx,
            # Trajectory length
            "episode_length":    len(frames),
            # State statistics (dimension-agnostic: compute L2 norm per frame, then aggregate)
            "state_mean_norm":   float(np.mean(np.linalg.norm(states, axis=1))),
            "state_std_norm":    float(np.std (np.linalg.norm(states, axis=1))),
            # Action statistics (same approach)
            "action_mean_norm":  float(np.mean(np.linalg.norm(actions, axis=1))),
            "action_std_norm":   float(np.std (np.linalg.norm(actions, axis=1))),
            # Reward statistics
            "avg_reward":        float(np.mean(rewards)),
            "max_reward":        float(np.max(rewards)),
            "reward_monotone":   float(np.corrcoef(range(len(rewards)), rewards)[0,1]
                                       if len(rewards) > 2 else 0),
        })
    return pd.DataFrame(episodes_data)

In [9]:
print("=== TEST 5: Aggregated Feature Extraction ===")

# Test extraction on each dataset
all_dfs = []
for name, ds in loaded.items():
    short = name.split("/")[1]
    df = extract_episode_features(ds, short, n_episodes=5)
    check(f"{short[:35]} extraction succeeded", len(df) > 0, f"{len(df)} episodes")
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
check("Merged DataFrame without errors", len(combined) > 0, f"total {len(combined)} episodes")
check("No NaN values", combined.isnull().sum().sum() == 0,
      f"NaN count: {combined.isnull().sum().sum()}")
check("Has source column", "source" in combined.columns)
check("Contains 8 data sources", combined["source"].nunique() == len(loaded),
      f"actual: {combined['source'].nunique()}")

=== TEST 5: Aggregated Feature Extraction ===
  ✅ PASS  pusht extraction succeeded  [4 episodes]
  ✅ PASS  xarm_lift_medium extraction succeeded  [5 episodes]
  ✅ PASS  xarm_lift_medium_replay extraction succeeded  [5 episodes]
  ✅ PASS  xarm_push_medium extraction succeeded  [5 episodes]
  ✅ PASS  xarm_push_medium_replay extraction succeeded  [5 episodes]
  ✅ PASS  berkeley_autolab_ur5 extraction succeeded  [5 episodes]
  ✅ PASS  columbia_cairlab_pusht_real extraction succeeded  [2 episodes]
  ✅ PASS  nyu_door_opening_surprising_effecti extraction succeeded  [5 episodes]
  ✅ PASS  Merged DataFrame without errors  [total 36 episodes]
  ✅ PASS  No NaN values  [NaN count: 0]
  ✅ PASS  Has source column
  ✅ PASS  Contains 8 data sources  [actual: 8]


---
## TEST 6: Per-Dataset Median Quality Labels

In [10]:
print("=== TEST 6: Quality Labels (per-dataset median) ===")

# Use per-dataset median as threshold (handles xarm_push negative rewards)
combined["quality"] = (
    combined["max_reward"]
    > combined.groupby("source")["max_reward"].transform("median")
).astype(int)

check("quality column contains only 0/1", set(combined["quality"].unique()).issubset({0, 1}))
check("Both positive and negative samples exist", combined["quality"].nunique() == 2,
      f"unique values: {sorted(combined['quality'].unique())}")

for source in combined["source"].unique():
    sub = combined[combined["source"] == source]
    rate = sub["quality"].mean()
    print(f"  {source:50s}  high quality rate: {rate:.1%}  ({sub['quality'].sum()}/{len(sub)})")

=== TEST 6: Quality Labels (per-dataset median) ===
  ✅ PASS  quality column contains only 0/1
  ✅ PASS  Both positive and negative samples exist  [unique values: [np.int64(0), np.int64(1)]]
  pusht                                               high quality rate: 50.0%  (2/4)
  xarm_lift_medium                                    high quality rate: 40.0%  (2/5)
  xarm_lift_medium_replay                             high quality rate: 40.0%  (2/5)
  xarm_push_medium                                    high quality rate: 40.0%  (2/5)
  xarm_push_medium_replay                             high quality rate: 40.0%  (2/5)
  berkeley_autolab_ur5                                high quality rate: 0.0%  (0/5)
  columbia_cairlab_pusht_real                         high quality rate: 0.0%  (0/2)
  nyu_door_opening_surprising_effectiveness           high quality rate: 0.0%  (0/5)


---
## TEST 7: Sim vs Real Grouping

In [11]:
print("=== TEST 7: Sim vs Real Grouping ===")

sim_sources = {n.split("/")[1] for n in SIM_DATASETS if n in loaded}
real_sources = {n.split("/")[1] for n in REAL_DATASETS if n in loaded}

combined["is_real"] = combined["source"].isin(real_sources).astype(int)

check("Simulation data exists", combined[combined["is_real"] == 0]["source"].nunique() > 0,
      f"{sim_sources & set(combined['source'].unique())}")
check("Real robot data exists", combined[combined["is_real"] == 1]["source"].nunique() > 0,
      f"{real_sources & set(combined['source'].unique())}")

# PushT sim vs real comparison
pusht_sim  = "pusht"
pusht_real = "columbia_cairlab_pusht_real"
both_exist = (pusht_sim in combined["source"].values and
              pusht_real in combined["source"].values)
check("PushT Sim/Real comparison data both exist", both_exist)

=== TEST 7: Sim vs Real Grouping ===
  ✅ PASS  Simulation data exists  [{'xarm_push_medium', 'xarm_lift_medium', 'xarm_lift_medium_replay', 'pusht', 'xarm_push_medium_replay'}]
  ✅ PASS  Real robot data exists  [{'berkeley_autolab_ur5', 'columbia_cairlab_pusht_real', 'nyu_door_opening_surprising_effectiveness'}]
  ✅ PASS  PushT Sim/Real comparison data both exist


---
## Test Summary

In [12]:
print("=" * 60)
print("Test Summary")
print("=" * 60)
passed = sum(1 for r in results if r[0] == PASS)
failed = sum(1 for r in results if r[0] == FAIL)
print(f"Passed: {passed}   Failed: {failed}   Total: {len(results)}")

if failed > 0:
    print("\nFailed items:")
    for r in results:
        if r[0] == FAIL:
            print(f"  {r[0]} {r[1]}: {r[2]}")

print("\n=== Sample Episode Features Preview ===")
combined[["source","episode_index","episode_length",
          "state_mean_norm","action_std_norm",
          "max_reward","quality","is_real"]]

Test Summary
Passed: 92   Failed: 0   Total: 92

=== Sample Episode Features Preview ===


,source,episode_index,episode_length,state_mean_norm,action_std_norm,max_reward,quality,is_real
0,pusht,0,161,367.160941,85.857258,0.873257,0,0
1,pusht,1,118,376.841510,69.251354,0.900152,0,0
2,pusht,2,141,277.458500,68.368104,0.908086,1,0
3,pusht,3,159,433.283511,66.627775,0.920348,1,0
4,xarm_lift_medium,0,25,1.845761,0.333524,1.175537,1,0
5,xarm_lift_medium,1,25,1.688873,0.279536,1.173541,0,0
6,xarm_lift_medium,2,25,1.758848,0.333030,1.168366,0,0
7,xarm_lift_medium,3,25,1.717295,0.304883,1.178847,1,0
8,xarm_lift_medium,4,25,1.741987,0.360627,1.165509,0,0
9,xarm_lift_medium_replay,0,25,1.640920,0.319086,-0.003770,1,0
